# Task 2: Exploratory Analysis
Data: clean_data.csv from Task 1. Salary is assumed to be monthly (INR). The budget table is sample data.

In [64]:
import numpy as np
import pandas as pd

df = pd.read_csv(r"C:\Users\LENOVO\Desktop\Python\week1_day5\clean_data.csv")
print(df.shape)
df

(13, 6)


,id,name,department,salary,city,salary_outlier
0,1,Arun,Sales,30000,Chennai,False
1,2,Priya,HR,45000,Chennai,False
2,3,Rahul,IT,60000,Bangalore,False
3,4,Divya,IT,75000,Chennai,False
4,5,Karthik,Finance,90000,Mumbai,False
5,6,Meena,Sales,52000,Bangalore,False
6,7,Suresh,IT,68000,Chennai,False
7,8,Anitha,HR,48000,Mumbai,False
8,10,Rahul,IT,60000,Bangalore,False
9,11,Kavya,Sales,38000,Chennai,False


In [65]:
## 1. Summary statistics

In [66]:
df["salary"].agg(["mean", "median", "min", "max", "std"]).round(2).to_frame("salary")

,salary
mean,59769.23
median,58000.00
min,30000.00
max,98000.00
std,19318.19


In [67]:
## 2. Group comparison with groupby().agg()

In [68]:
by_dept = (df.groupby("department")["salary"]
             .agg(employees="count", mean="mean", median="median", min="min", max="max", std="std")
             .round(2).sort_values("mean", ascending=False))
by_dept

,employees,mean,median,min,max,std
department,,,,,,
Finance,2,94000.00,94000.0,90000,98000,5656.85
IT,6,62666.67,60000.0,55000,75000,7420.69
HR,2,46500.00,46500.0,45000,48000,2121.32
Sales,3,40000.00,38000.0,30000,52000,11135.53


In [69]:
by_city = (df.groupby("city")["salary"]
             .agg(employees="count", mean="mean", median="median", min="min", max="max", std="std")
             .round(2).sort_values("mean", ascending=False))
by_city

,employees,mean,median,min,max,std
city,,,,,,
Mumbai,3,64333.33,55000.0,48000,90000,22501.85
Chennai,7,58857.14,58000.0,30000,98000,23582.68
Bangalore,3,57333.33,60000.0,52000,60000,4618.80


In [70]:
## 3. Merge with a department budget table (sample data)

In [71]:
budget = pd.DataFrame({
    "department": ["Sales", "HR", "IT", "Finance"],
    "annual_budget": [1_800_000, 1_200_000, 5_000_000, 2_500_000],
})
merged = pd.merge(df, budget, on="department", how="left")
print("Rows:", len(df), "->", len(merged), "| unmatched:", merged["annual_budget"].isna().sum())
merged.head()

Rows: 13 -> 13 | unmatched: 0


,id,name,department,salary,city,salary_outlier,annual_budget
0,1,Arun,Sales,30000,Chennai,False,1800000
1,2,Priya,HR,45000,Chennai,False,1200000
2,3,Rahul,IT,60000,Bangalore,False,5000000
3,4,Divya,IT,75000,Chennai,False,5000000
4,5,Karthik,Finance,90000,Mumbai,False,2500000


In [72]:
payroll = (merged.groupby("department")
           .agg(employees=("id", "count"),
                monthly_payroll=("salary", "sum"),
                annual_budget=("annual_budget", "first")))
payroll["annual_payroll"] = payroll["monthly_payroll"] * 12
payroll["budget_used_pct"] = (payroll["annual_payroll"] / payroll["annual_budget"] * 100).round(1)
payroll.sort_values("budget_used_pct", ascending=False)

,employees,monthly_payroll,annual_budget,annual_payroll,budget_used_pct
department,,,,,
HR,2,93000,1200000,1116000,93.0
Finance,2,188000,2500000,2256000,90.2
IT,6,376000,5000000,4512000,90.2
Sales,3,120000,1800000,1440000,80.0


In [73]:
## 4. Correlations and top 3 relationships

In [74]:
num = merged[["salary", "annual_budget"]].copy()
num["dept_headcount"] = merged.groupby("department")["id"].transform("count")
num["dept_avg_salary"] = merged.groupby("department")["salary"].transform("mean")
corr = num.corr().round(3)
corr

,salary,annual_budget,dept_headcount,dept_avg_salary
salary,1.000,0.305,0.023,0.935
annual_budget,0.305,1.000,0.953,0.326
dept_headcount,0.023,0.953,1.000,0.025
dept_avg_salary,0.935,0.326,0.025,1.000


In [75]:
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
pairs = upper.stack()
pairs.reindex(pairs.abs().sort_values(ascending=False).index).head(3).rename("correlation").to_frame()

,,correlation
annual_budget,dept_headcount,0.953
salary,dept_avg_salary,0.935
annual_budget,dept_avg_salary,0.326


Note: with 13 rows these correlations are only indicative. salary vs dept_avg_salary is high by construction, and budget vs headcount comes from the sample budgets. Correlation does not prove causation.

In [76]:
## 5. Business questions

In [77]:
q1 = by_dept["mean"].idxmax()
print(f"Q1. Highest average salary: {q1}, {by_dept.loc[q1, 'mean']:,.0f} per month")

Q1. Highest average salary: Finance, 94,000 per month


In [78]:
q2 = by_city["employees"].idxmax()
print(f"Q2. City with most employees: {q2}, {by_city.loc[q2, 'employees']} ({by_city.loc[q2, 'employees'] / len(df):.0%} of staff)")

Q2. City with most employees: Chennai, 7 (54% of staff)


In [79]:
share = (df.groupby("department")["salary"].sum() / df["salary"].sum() * 100).round(1).sort_values(ascending=False)
print("Q3. Share of total monthly payroll (%):")
print(share)

Q3. Share of total monthly payroll (%):
department
IT         48.4
Finance    24.2
Sales      15.4
HR         12.0
Name: salary, dtype: float64


In [80]:
q4 = payroll["budget_used_pct"].idxmax()
print(f"Q4. Department using most of its budget: {q4}, {payroll.loc[q4, 'budget_used_pct']}%")

Q4. Department using most of its budget: HR, 93.0%


In [81]:
gap = by_dept["mean"].max() - by_dept["mean"].min()
med = df["salary"].median()
print(f"Q5. Gap between highest and lowest department average: {gap:,.0f} per month")
print(f"    Employees above the median ({med:,.0f}): {(df['salary'] > med).sum()} of {len(df)}")

Q5. Gap between highest and lowest department average: 54,000 per month
    Employees above the median (58,000): 6 of 13


In [82]:
## 6. Bonus: pivot table

In [83]:
pd.pivot_table(df, values="salary", index="department", columns="city",
               aggfunc="mean", margins=True, margins_name="Total", fill_value=0).round(0)

city,Bangalore,Chennai,Mumbai,Total
department,,,,
Finance,0.0,98000.0,90000.0,94000.0
HR,0.0,45000.0,48000.0,46500.0
IT,60000.0,67000.0,55000.0,62667.0
Sales,52000.0,34000.0,0.0,40000.0
Total,57333.0,58857.0,64333.0,59769.0


In [84]:
pd.pivot_table(df, values="id", index="department", columns="city",
               aggfunc="count", margins=True, margins_name="Total", fill_value=0)

city,Bangalore,Chennai,Mumbai,Total
department,,,,
Finance,0,1,1,2
HR,0,1,1,2
IT,2,3,1,6
Sales,1,2,0,3
Total,3,7,3,13
